# Deploy NPS Agent to RHOAI

This notebook deploys the NPS Agent ([`npsagent.py`](./npsagent.py)) to OpenShift AI with MLflow tracing.

## Prerequisites

- OpenShift cluster with RHOAI and MLflow
- `oc` CLI logged in
- An [OpenAI API key](https://platform.openai.com/api-keys)
- A [NPS API key](https://www.nps.gov/subjects/developer/get-started.htm)

In [ ]:
import os

## OpenAI Environment Variables
os.environ["OPENAI_API_KEY"] = ""
os.environ["OPENAI_BASE_URL"] = ""
os.environ["OPENAI_MODEL_NAME"] = ""

## NPS API Key
os.environ["NPS_API_KEY"] = ""

## Cluster-specific values (update these!)
NAMESPACE = "nps-agent-<yourname>"
MLFLOW_TRACKING_URI = "https://data-science-gateway.apps.<cluster>/mlflow/"
MLFLOW_WORKSPACE = NAMESPACE
MLFLOW_EXPERIMENT_NAME = "nps-agent"

# Check that required vars are set
required_vars = ["OPENAI_API_KEY", "OPENAI_BASE_URL", "OPENAI_MODEL_NAME", "NPS_API_KEY"]
if any(not os.getenv(var) for var in required_vars):
    raise ValueError("One or more required environment variables are not set, please set them above.")

## Step 1 — Create an OpenShift Project

Use `nps-agent-<yourname>` to avoid conflicts with other users on the same cluster.

In [ ]:
!oc new-project {NAMESPACE}

## Step 2 — Create the Secret with API Keys

In [ ]:
!oc create secret generic nps-agent-secrets \
  --from-literal=OPENAI_API_KEY="{os.getenv('OPENAI_API_KEY')}" \
  --from-literal=NPS_API_KEY="{os.getenv('NPS_API_KEY')}" \
  -n {NAMESPACE}

## Step 3 — Configure Environment and Apply the Template

The [`nps-agent.yaml`](./nps-agent.yaml) is an OpenShift Template. We use `oc process` to inject our parameters.

This creates:
- **BuildConfig** — s2i build from the `deploy` branch of this repo
- **ImageStream** — stores the built container image
- **Deployment** — runs the agent pod (MCP server + MLflow serve)
- **Service** — internal cluster networking
- **Route** — external HTTPS endpoint

In [ ]:
!oc process -f ./nps-agent.yaml \
  -p NAMESPACE="{NAMESPACE}" \
  -p MLFLOW_TRACKING_URI="{MLFLOW_TRACKING_URI}" \
  -p MLFLOW_WORKSPACE="{MLFLOW_WORKSPACE}" \
  -p MLFLOW_EXPERIMENT_NAME="{MLFLOW_EXPERIMENT_NAME}" \
  | oc apply -f -

## Step 4 — Wait for the s2i Build

The BuildConfig triggers automatically. Watch the build until you see **"Push successful"**.

In [ ]:
!oc logs -f build/nps-agent-1 -n {NAMESPACE}

## Step 5 — Set the MLflow Auth Token

The RHOAI Data Science Gateway requires an auth token. Set it from your current `oc` session.

> **Note:** `oc` tokens expire. Re-run this cell when you need to refresh.

In [ ]:
!oc set env deployment/nps-agent \
  MLFLOW_TRACKING_TOKEN="$(oc whoami -t)" \
  -n {NAMESPACE}

## Step 6 — Verify the Pod is Running

In [ ]:
!oc get pods -n {NAMESPACE}

## Step 7 — Get the Route URL

In [ ]:
import subprocess

ROUTE_HOST = subprocess.check_output(
    ["oc", "get", "route", "nps-agent", "-n", NAMESPACE, "-o", "jsonpath={.spec.host}"]
).decode().strip()

AGENT_URL = f"https://{ROUTE_HOST}"
print(f"Agent URL: {AGENT_URL}")

## Step 8 — Test the Agent

In [ ]:
import requests
from IPython.display import display, Markdown

payload = {
    "input": [
        {"role": "user", "content": "What national parks are in California?"}
    ]
}

resp = requests.post(
    f"{AGENT_URL}/invocations",
    headers={"Content-Type": "application/json"},
    json=payload,
)

print(f"Status: {resp.status_code}")
result = resp.json()
output_text = result.get("output", [{}])[0].get("text", str(result))
display(Markdown(output_text))

## Step 9 — View Traces in MLflow

Open your RHOAI MLflow UI and navigate to the `nps-agent` experiment. Every request is auto-traced.

## Rebuilding After Code Changes

Push changes to the `deploy` branch, then trigger a new build:

In [ ]:
!oc start-build nps-agent -n {NAMESPACE}
!oc logs -f build/nps-agent-2 -n {NAMESPACE}

## Cleanup

To delete everything and start from scratch:

In [ ]:
!oc delete project {NAMESPACE}